# Testing the `AdvisorWorkFlow _v1.3` n8n Workflow

Test cases are loaded from `unit_tests/advisor/*/test.json` (generated by the
unit-test generator).  Each file contains a pre-built `input` dict that is
posted verbatim to the webhook, and an `expectedOutput` dict used for
pass/fail comparison.

### What the webhook returns
```json
{
  "consultAcc": "...", "maxPayment": 0, "maxTerm": 0,
  "DebtSituation": "...", "refPlanID": null,
  "maxPaymentY2": null, "maxPaymentY3": null, "reConfirmMessage": ""
}
```

### URL note
- Test URL (n8n editor open): `{base}/webhook-test/{path}`
- Production URL (workflow Active): `{base}/webhook/{path}`

In [1]:
import json
from pathlib import Path

import pandas as pd
import math
import requests

pd.set_option('display.max_rows', 2000)
pd.set_option('display.max_columns', 1500)

In [2]:
from pathlib import Path

INPUT_DIR = Path("./final_test_case_gen/unit_tests/advisor")

for fp in INPUT_DIR.rglob("*.json"):
    with open(fp, "r", encoding="utf-8") as f:
        data = json.load(f)

    row = {
        "test_case_folder": fp.parent.name,
        "file_name": fp.name,
        "file_path": str(fp),

        "testId": data.get("testId"),
        "agentType": data.get("agentType"),
        "scenario": data.get("scenario"),
        "messageMode": data.get("messageMode"),
        "guardrailCategory": data.get("guardrailCategory"),
        "testDescription": data.get("testDescription"),

        "userMessage": get_nested(data, "input.userMessage"),
        "LastAImessage": get_nested(data, "input.LastAImessage"),
        "consultAcc": get_nested(data, "input.consultAcc"),
        "DebtSituation": get_nested(data, "input.DebtSituation"),
        "maxPayment": get_nested(data, "input.maxPayment"),
        "maxTerm": get_nested(data, "input.maxTerm"),

        "expected_route": get_nested(data, "expected.route_to") or data.get("expected_route"),
        "expected_json": json.dumps(data.get("expected", {}), ensure_ascii=False),
        "full_json": json.dumps(data, ensure_ascii=False),
    }

    rows.append(row)

## 1. Configuration

In [3]:
N8N_BASE_URL = "https://alphamakeathon-automation.arisetech.dev"  # <-- change me
WEBHOOK_PATH = "b7607735-bac3-4965-8c68-2ed0ef38aec4"
USE_TEST_URL = False  # True -> use the "Listen for test event" URL instead
# TEST_CASES_XLSX = "advisor_test_cases_100_full.xlsx"  # <-- upload this alongside the notebook

def get_webhook_url() -> str:
    prefix = "webhook-test" if USE_TEST_URL else "webhook"
    return f"{N8N_BASE_URL}/{prefix}/{WEBHOOK_PATH}"


get_webhook_url()

'https://alphamakeathon-automation.arisetech.dev/webhook/b7607735-bac3-4965-8c68-2ed0ef38aec4'

## 2. Load test cases from JSON

In [4]:
def _find_project_root() -> Path:
    p = Path.cwd()
    while p != p.parent:
        if (p / "unit_tests").is_dir():
            return p
        p = p.parent
    raise RuntimeError("Could not find project root (directory containing 'unit_tests/')")


def load_unit_tests(agent_type: str) -> list[dict]:
    root = _find_project_root()
    test_dir = root / "unit_tests" / agent_type
    tests = []
    for tc_dir in sorted(test_dir.iterdir()):
        test_file = tc_dir / "test.json"
        if test_file.is_file():
            with open(test_file, encoding="utf-8") as f:
                tests.append(json.load(f))
    print(f"Loaded {len(tests)} test cases from {test_dir}")
    return tests


TEST_CASES = load_unit_tests("advisor")
for tc in TEST_CASES:
    print(f"  {tc['testId']}: {tc['testDescription']}")

Loaded 150 test cases from /Users/IF668129/Downloads/DebtMind-test-case-generator-main/final_test_case_gen/unit_tests/advisor
  TC-0001: [strict] NEGOTIATE_PAYMENT route_to='advisor' — advisor extracts: consultAcc='xxxxxx503620, xxxxxx503619', maxPayment=700
  TC-0002: [adversarial] NEGOTIATE_TERM route_to='advisor' — advisor extracts: consultAcc='xxxxxx500362,xxxxxx500364,xxxxxx500363', maxPayment=46000
  TC-0003: [adversarial] NEGOTIATE_TERM route_to='advisor' — advisor extracts: consultAcc='xxxxxx500362,xxxxxx500364,xxxxxx500363', maxPayment=46000
  TC-0004: [strict] NEGOTIATE_TERM route_to='advisor' — advisor extracts: consultAcc='xxxxxx503379', maxPayment=0
  TC-0005: [strict] AFTER_OFFER_TEXT route_to='advisor' — advisor extracts: consultAcc='xxxxxx500362,xxxxxx500364,xxxxxx500363', maxPayment=46000
  TC-0006: [strict] AFTER_OFFER_TEXT route_to='advisor' — advisor extracts: consultAcc='', maxPayment=None
  TC-0007: [strict] NEGOTIATE_PAYMENT route_to='advisor' — advisor extracts:

## 3. Webhook caller

In [5]:
def call_advisor(payload: dict, timeout: int = 60) -> dict:
    """POST the pre-built input dict to the Advisor webhook and return parsed JSON."""
    url = get_webhook_url()
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

## 4. Run a single example

Inspect one test case and confirm connectivity before running the full suite.

In [6]:
sample_tc = TEST_CASES[0]
print(f"Test: {sample_tc['testId']} — {sample_tc['testDescription']}")
print("\nInput payload:")
print(json.dumps(sample_tc["input"], ensure_ascii=False, indent=2))
print("\nExpected output:")
print(json.dumps(sample_tc["expectedOutput"], ensure_ascii=False, indent=2))

result = call_advisor(sample_tc["input"])
print("\nActual output:")
print(json.dumps(result, ensure_ascii=False, indent=2))

Test: TC-0001 — [strict] NEGOTIATE_PAYMENT route_to='advisor' — advisor extracts: consultAcc='xxxxxx503620, xxxxxx503619', maxPayment=700

Input payload:
{
  "consultAcc": "",
  "DebtSituation": "DebtBurden",
  "maxPayment": 700,
  "maxTerm": 120,
  "refPlanID": "TDR0420260621103821",
  "maxPaymentY2": null,
  "maxPaymentY3": null,
  "narrative": "The customer has reviewed the proposed debt restructuring plans and finds the monthly payment of 966 THB to be too high. They have countered with a request for a plan with a monthly payment of 700 THB. This is being routed back to the advisor to see if a revised plan is possible.",
  "accText": "[{\"account_number\": \"1xxxxxxx0804\", \"port\": \"สินเชื่ออเนกประสงค์ 5 Plus\", \"creditlimit\": 15000.0, \"os\": 7096.95, \"installment\": 800.0, \"remaining_term\": 10}, {\"account_number\": \"1xxxxxxx0803\", \"port\": \"สินเชื่อส่วนบุคคล\", \"creditlimit\": 150000.0, \"os\": 33518.6, \"installment\": 2500.0, \"remaining_term\": 16}, {\"account_nu

## 5. Run all test cases and compare against expectedOutput

In [7]:
COMPARE_KEYS = [
    "consultAcc", "DebtSituation", "maxPayment", "maxTerm",
    "refPlanID", "maxPaymentY2", "maxPaymentY3", "reConfirmMessage",
]

rows = []
for tc in TEST_CASES:
    try:
        result = call_advisor(tc["input"])
        expected = tc["expectedOutput"]
        mismatches = [k for k in COMPARE_KEYS if expected.get(k) != result.get(k)]
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "pass": len(mismatches) == 0,
            "mismatched_fields": ", ".join(mismatches) if mismatches else "",
            **{f"exp_{k}": expected.get(k) for k in COMPARE_KEYS},
            **{f"act_{k}": result.get(k) for k in COMPARE_KEYS},
            "error": None,
        })
    except requests.exceptions.RequestException as exc:
        rows.append({
            "testId": tc["testId"],
            "scenario": tc["scenario"],
            "messageMode": tc["messageMode"],
            "pass": False,
            "mismatched_fields": "",
            **{f"exp_{k}": None for k in COMPARE_KEYS},
            **{f"act_{k}": None for k in COMPARE_KEYS},
            "error": str(exc),
        })

results_df = pd.DataFrame(rows)
summary_cols = ["testId", "scenario", "messageMode", "pass", "mismatched_fields", "error"]
print(f"Passed: {results_df['pass'].sum()}/{len(results_df)}")
results_df[summary_cols]

Passed: 109/150


,testId,scenario,messageMode,pass,mismatched_fields,error
0,TC-0001,NEGOTIATE_PAYMENT,strict,True,,None
1,TC-0002,NEGOTIATE_TERM,adversarial,True,,None
2,TC-0003,NEGOTIATE_TERM,adversarial,True,,None
3,TC-0004,NEGOTIATE_TERM,strict,True,,None
4,TC-0005,AFTER_OFFER_TEXT,strict,True,,None
5,TC-0006,AFTER_OFFER_TEXT,strict,False,"consultAcc, DebtSituation, maxPayment, maxTerm...",None
6,TC-0007,NEGOTIATE_PAYMENT,strict,False,refPlanID,None
7,TC-0008,MULTI_TURN,strict,True,,None
8,TC-0009,AFTER_OFFER_TEXT,strict,False,"consultAcc, reConfirmMessage",None
9,TC-0010,AFTER_OFFER_TEXT,creative,True,,None


In [8]:
results_df.head()

,testId,scenario,messageMode,pass,mismatched_fields,exp_consultAcc,exp_DebtSituation,exp_maxPayment,exp_maxTerm,exp_refPlanID,exp_maxPaymentY2,exp_maxPaymentY3,exp_reConfirmMessage,act_consultAcc,act_DebtSituation,act_maxPayment,act_maxTerm,act_refPlanID,act_maxPaymentY2,act_maxPaymentY3,act_reConfirmMessage,error
0,TC-0001,NEGOTIATE_PAYMENT,strict,True,,"xxxxxx503620, xxxxxx503619",DebtBurden,700.0,120.0,NaN,None,None,,"xxxxxx503620, xxxxxx503619",DebtBurden,700.0,120.0,NaN,None,None,,None
1,TC-0002,NEGOTIATE_TERM,adversarial,True,,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,60.0,TDR0120260621102006,None,None,,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,60.0,TDR0120260621102006,None,None,,None
2,TC-0003,NEGOTIATE_TERM,adversarial,True,,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,60.0,NaN,None,None,,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,60.0,NaN,None,None,,None
3,TC-0004,NEGOTIATE_TERM,strict,True,,xxxxxx503379,DebtBurden,0.0,75.0,TDR0220260621085331,None,None,CLARIFY_LONG_TERM,xxxxxx503379,DebtBurden,0.0,75.0,TDR0220260621085331,None,None,CLARIFY_LONG_TERM,None
4,TC-0005,AFTER_OFFER_TEXT,strict,True,,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,360.0,TDR0120260621102006,None,None,CLARIFY_HIGH_INSTALLMENT,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,360.0,TDR0120260621102006,None,None,CLARIFY_HIGH_INSTALLMENT,None


In [27]:
results_df.shape

(150, 22)

In [24]:
results_df[["pass"]].value_counts()

pass 
True     109
False     41
Name: count, dtype: int64

In [29]:
df_true = results_df[results_df["pass"] == True]
df_false = results_df[results_df["pass"] == False]

df_sample = pd.concat([
    df_true.sample(n=85, random_state=42),
    df_false.sample(n=15, random_state=42)
])

# สุ่มลำดับแถว
df_sample = df_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(df_sample["pass"].value_counts())
print(df_sample.shape)

pass
True     85
False    15
Name: count, dtype: int64
(100, 22)


In [30]:
# true_cases = results_df[results_df["pass"] == True]
# false_cases = results_df[results_df["pass"] == False].sample(n=17, random_state=42)

# selected_df = pd.concat([true_cases, false_cases]).reset_index(drop=True)

# print(f"Total: {len(selected_df)}")
# print(selected_df["pass"].value_counts())

## compare

In [31]:
df_sample.head()

,testId,scenario,messageMode,pass,mismatched_fields,exp_consultAcc,exp_DebtSituation,exp_maxPayment,exp_maxTerm,exp_refPlanID,exp_maxPaymentY2,exp_maxPaymentY3,exp_reConfirmMessage,act_consultAcc,act_DebtSituation,act_maxPayment,act_maxTerm,act_refPlanID,act_maxPaymentY2,act_maxPaymentY3,act_reConfirmMessage,error
0,TC-0095,AFTER_OFFER_TEXT,creative,True,,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103031,None,None,CLARIFY_HIGH_INSTALLMENT,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103031,None,None,CLARIFY_HIGH_INSTALLMENT,None
1,TC-0129,NEGOTIATE_PAYMENT,adversarial,True,,"xxxxxx503620,xxxxxx503619",DebtBurden,700.0,360.0,TDR0220260621102914,None,None,,"xxxxxx503620,xxxxxx503619",DebtBurden,700.0,360.0,TDR0220260621102914,None,None,,None
2,TC-0070,MULTI_TURN,strict,True,,,DebtBurden,0.0,120.0,NaN,None,None,,,DebtBurden,0.0,120.0,NaN,None,None,,None
3,TC-0046,MULTI_TURN,strict,True,,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103712,None,None,CLARIFY_HIGH_INSTALLMENT,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103712,None,None,CLARIFY_HIGH_INSTALLMENT,None
4,TC-0112,STAFF_REQUEST,strict,True,,,DebtBurden,0.0,360.0,NaN,None,None,,,DebtBurden,0.0,360.0,NaN,None,None,,None


In [32]:
COMPARE_FIELDS = ["consultAcc", "DebtSituation", "maxPayment", "maxTerm", "refPlanID", "maxPaymentY2", "maxPaymentY3", "reConfirmMessage"]
NUMERIC_FIELDS = {"maxPayment", "maxTerm", "maxPaymentY2", "maxPaymentY3"}
TOL = 1e-8

In [34]:
results_df2 = df_sample.copy()

In [35]:
# แปลง NaN เป็น ''
def is_empty(v):
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except:
        pass
    return str(v).strip() in ["", "None", "nan", "NaN"]


# แปลง consultAcc เป็น set
def normalize_consult_acc(v):
    if is_empty(v):
        return frozenset()

    return frozenset(
        x.strip()
        for x in str(v).replace("，", ",").split(",")
        if x.strip()
    )

# แปลงค่า
def normalize(v, field):
    if is_empty(v):
        return frozenset() if field == "consultAcc" else ""

    if field in NUMERIC_FIELDS:
        try:
            return float(v)
        except:
            return ""

    if field == "consultAcc":
        return normalize_consult_acc(v)

    return str(v).strip()

# เปรียบเทียบค่า expected vs actual
def compare_value(exp, act, field):
    exp_n = normalize(exp, field)
    act_n = normalize(act, field)

    if field in NUMERIC_FIELDS:
        if exp_n == "" and act_n == "":
            return True
        if exp_n == "" or act_n == "":
            return False
        return math.isclose(exp_n, act_n, abs_tol=TOL)

    return exp_n == act_n


# เปรียบเทียบแต่ละ field และสร้างคอลัมน์ pass_{field}
for field in COMPARE_FIELDS:
    results_df2[f"pass_{field}"] = results_df2.apply(
        lambda row: compare_value(
            row[f"exp_{field}"],
            row[f"act_{field}"],
            field
        ),
        axis=1
    )

pass_cols = [f"pass_{field}" for field in COMPARE_FIELDS]

# pass = True เมื่อทุก field ผ่านหมด
results_df2["pass_check"] = results_df2[pass_cols].all(axis=1)

# รวมชื่อ field ที่ไม่ผ่านไว้ใน string เดียว เพื่อง่ายต่อการ debug
results_df2["mismatched_fields"] = results_df2.apply(lambda row: ",".join([field for field in COMPARE_FIELDS if not row[f"pass_{field}"]]), axis=1)

In [36]:
results_df2.head()

,testId,scenario,messageMode,pass,mismatched_fields,exp_consultAcc,exp_DebtSituation,exp_maxPayment,exp_maxTerm,exp_refPlanID,exp_maxPaymentY2,exp_maxPaymentY3,exp_reConfirmMessage,act_consultAcc,act_DebtSituation,act_maxPayment,act_maxTerm,act_refPlanID,act_maxPaymentY2,act_maxPaymentY3,act_reConfirmMessage,error,pass_consultAcc,pass_DebtSituation,pass_maxPayment,pass_maxTerm,pass_refPlanID,pass_maxPaymentY2,pass_maxPaymentY3,pass_reConfirmMessage,pass_check
0,TC-0095,AFTER_OFFER_TEXT,creative,True,,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103031,None,None,CLARIFY_HIGH_INSTALLMENT,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103031,None,None,CLARIFY_HIGH_INSTALLMENT,None,True,True,True,True,True,True,True,True,True
1,TC-0129,NEGOTIATE_PAYMENT,adversarial,True,,"xxxxxx503620,xxxxxx503619",DebtBurden,700.0,360.0,TDR0220260621102914,None,None,,"xxxxxx503620,xxxxxx503619",DebtBurden,700.0,360.0,TDR0220260621102914,None,None,,None,True,True,True,True,True,True,True,True,True
2,TC-0070,MULTI_TURN,strict,True,,,DebtBurden,0.0,120.0,NaN,None,None,,,DebtBurden,0.0,120.0,NaN,None,None,,None,True,True,True,True,True,True,True,True,True
3,TC-0046,MULTI_TURN,strict,True,,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103712,None,None,CLARIFY_HIGH_INSTALLMENT,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103712,None,None,CLARIFY_HIGH_INSTALLMENT,None,True,True,True,True,True,True,True,True,True
4,TC-0112,STAFF_REQUEST,strict,True,,,DebtBurden,0.0,360.0,NaN,None,None,,,DebtBurden,0.0,360.0,NaN,None,None,,None,True,True,True,True,True,True,True,True,True


In [37]:
results_df2[["pass_check"]].value_counts()

pass_check
True          91
False          9
Name: count, dtype: int64

In [39]:
results_df2[results_df2["pass_check"] == False]

,testId,scenario,messageMode,pass,mismatched_fields,exp_consultAcc,exp_DebtSituation,exp_maxPayment,exp_maxTerm,exp_refPlanID,exp_maxPaymentY2,exp_maxPaymentY3,exp_reConfirmMessage,act_consultAcc,act_DebtSituation,act_maxPayment,act_maxTerm,act_refPlanID,act_maxPaymentY2,act_maxPaymentY3,act_reConfirmMessage,error,pass_consultAcc,pass_DebtSituation,pass_maxPayment,pass_maxTerm,pass_refPlanID,pass_maxPaymentY2,pass_maxPaymentY3,pass_reConfirmMessage,pass_check
14,TC-0141,AFTER_OFFER_TEXT,strict,False,"consultAcc,DebtSituation,maxPayment,maxTerm",,NaN,NaN,NaN,NaN,None,None,,xxxxxx503619,DebtBurden,500.0,120.0,NaN,None,None,,None,False,False,False,False,True,True,True,True,False
27,TC-0052,NEGOTIATE_TERM,adversarial,False,refPlanID,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,60.0,TDR0120260621102006,None,None,,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,60.0,NaN,None,None,,None,True,True,True,True,False,True,True,True,False
32,TC-0125,MULTI_TURN,strict,False,"consultAcc,DebtSituation,maxPayment,maxTerm","xxxxxx501645,xxxxxx501646",DebtBurden,800.0,360.0,NaN,None,None,,NaN,NaN,NaN,NaN,NaN,None,None,NaN,None,False,False,False,False,True,True,True,True,False
34,TC-0142,MULTI_TURN,strict,False,"consultAcc,DebtSituation,maxPayment,maxTerm,re...",,NaN,NaN,NaN,NaN,None,None,,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103712,None,None,CLARIFY_HIGH_INSTALLMENT,None,False,False,False,False,False,True,True,False,False
61,TC-0071,MULTI_TURN,strict,False,refPlanID,xxxxxx503619,DebtBurden,500.0,120.0,TDR0220260621103712,None,None,,xxxxxx503619,DebtBurden,500.0,120.0,NaN,None,None,,None,True,True,True,True,False,True,True,True,False
69,TC-0084,AFTER_OFFER_TEXT,adversarial,False,"consultAcc,DebtSituation,maxPayment,maxTerm","xxxxxx501645,xxxxxx501646",DebtBurden,800.0,360.0,NaN,None,None,,NaN,NaN,NaN,NaN,NaN,None,None,NaN,None,False,False,False,False,True,True,True,True,False
74,TC-0034,AFTER_OFFER_TEXT,strict,False,refPlanID,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,37000.0,360.0,,None,None,,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,37000.0,360.0,TDR0120260621102006,None,None,,None,True,True,True,True,False,True,True,True,False
89,TC-0031,NEGOTIATE_TERM,strict,False,reConfirmMessage,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,195.0,TDR0120260621102006,None,None,,"xxxxxx500362,xxxxxx500364,xxxxxx500363",DebtBurden,46000.0,195.0,TDR0120260621102006,None,None,CLARIFY_LONG_TERM,None,True,True,True,True,True,True,True,False,False
90,TC-0096,NEGOTIATE_PAYMENT,adversarial,False,"consultAcc,refPlanID",,DebtBurden,1000.0,120.0,TDR0320260621085331,None,None,,xxxxxx503379,DebtBurden,1000.0,120.0,NaN,None,None,,None,False,True,True,True,False,True,True,True,False


## Notes

- This workflow is **Active**, so the production URL is
  `{base_url}/webhook/b7607735-bac3-4965-8c68-2ed0ef38aec4`.
- The webhook only returns the raw extractor output — to test the full
  advisor pipeline (offer-engine, clarify reply), invoke as a sub-workflow.
- Add more test cases by running the unit-test generator; they are picked up
  automatically the next time this notebook runs.
- If the n8n webhook requires authentication, add `auth=` or `headers=` to
  `requests.post` inside `call_advisor`.

In [19]:
## review match